# Transilien SNCF Challenge

## Introduction

This notebook was developed to address a short-term train delay prediction challenge using real-world data from SNCF Transilien.

The goal is to predict the difference between theoretical and actual waiting times at a station, two stops upstream. The target variable, p0q0, represents this delay in minutes.

To solve this problem, I structured the pipeline into several key stages: data loading and cleaning, exploratory analysis, feature engineering, outlier removal, model training, and finally, result. The final predictions are designed to match the format of a real-world deployment pipeline.

Throughout this notebook, I experiment with different models and optimization strategies to find the best-performing solution in terms of accuracy and interpretability

# Imports & Functions

This section gathers all the necessary imports and reusable functions used throughout the notebook.

In [1]:
import pandas as pd
import numpy as np
from functools import partial
import optuna.visualization as vis
import plotly.express as px
# from itertools import combinations ## Import for the search of the best set of variables

from sklearn.metrics import mean_absolute_error

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from sklearn.ensemble import HistGradientBoostingRegressor
import xgboost as xgb
import optuna

Helper functions are also defined for tasks like outlier removal and the objective function used by Optuna to optimize the XGBoost model.
Designing modular functions here made the rest of the workflow cleaner and more maintainable.

In [ ]:
############ OLD ############
def remove_top_1_percent_outliers(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    df_clean = df.copy()
    for col in features:
        lower_bound = np.percentile(df_clean[col], 1)
        upper_bound = np.percentile(df_clean[col], 99)
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
    return df_clean
##############################

In [2]:
def remove_outliers(df: pd.DataFrame, target: pd.DataFrame, lower_threshold: float=0.01, upper_threshold: float=0.001) -> tuple[pd.DataFrame]:
    """Removes the extreme outliers

    Args:
        df (pd.DataFrame): The dataframe to adapt based on the target
        target (pd.DataFrame): The Data Frame to apply the outlier feature
        threshold (float, optional): The outlier quantile. Defaults to 0.012.

    Returns:
        tuple[pd.DataFrame]: The modified dataframe
    """
    lower_bound = target.quantile(lower_threshold)
    upper_bound = target.quantile(upper_threshold)
    
    mask = (target >= lower_bound) & (target <= upper_bound)
    return df[mask], target[mask]

def objective(trial, X_train_clean: pd.DataFrame, Y_train_clean: pd.DataFrame, X_val: pd.DataFrame, y_val: pd.DataFrame):
    """Function to optimize the parameters of the XGBoost model

    Args:
        trial (_type_): _description_
        X_train_clean (pd.DataFrame): The training data set with all the variables
        Y_train_clean (pd.DataFrame): The training data with results
        X_val (pd.DataFrame): The validation training data 
        y_val (pd.DataFrame): The validation training data with results
    """
    params = {
        "max_depth": trial.suggest_int("max_depth", 4, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
        "objective": "reg:squarederror",
        "eval_metric": "mae"
    }

    dtrain = xgb.DMatrix(X_train_clean, label=Y_train_clean)
    dval = xgb.DMatrix(X_val, label=y_val)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        evals=[(dval, "validation")],
        early_stopping_rounds=50,
        verbose_eval=False
    )

    preds = model.predict(dval)
    preds = preds.round().astype(int)
    mae = mean_absolute_error(y_val, preds)
    return mae

def remove_high_w_model(X: pd.DataFrame, y: pd.DataFrame, threshold: float=0.98) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Train a simple and fast model to detect the highest outliers with a threshold

    Args:
        X (pd.DataFrame): The dataframe to adapt based on the target
        y (pd.DataFrame): The Data Frame to apply the outlier feature
        threshold (float, optional): The outlier quantile. Defaults to 0.98.

    Returns:
        tuple[pd.DataFrame]: The modified dataframe
    """
    model = xgb.XGBRegressor(n_estimators=100, max_depth=4, n_jobs=-1)
    model.fit(X, y)
    preds = model.predict(X)
    errors = abs(preds - y)

    error_thresh = np.quantile(errors, threshold)
    mask = errors <= error_thresh

    return X[mask], y[mask]

# Read files & plot basic information

In this part, the relevant datasets are loaded into memory: the train/test features (x_train_file, x_test_file), the target variable (y_train_file), and a sample submission.

A first exploratory visualization is generated using Plotly to display the frequency of each p0q0 value.
Since p0q0 represents a discrete delay difference in minutes, this helped us understand the distribution of delays and revealed the data imbalance around zero, which is useful for guiding modeling strategies.

In [3]:
x_test_file = pd.read_csv(r"x_test_final.csv")
x_train_file = pd.read_csv(r"x_train_final.csv")
y_sample = pd.read_csv(r"y_sample_final.csv")
y_train_file = pd.read_csv(r"y_train_final_j5KGWWK.csv")

In [4]:
# Get the frequency of each unique value of p0q0
target_counts = y_train_file["p0q0"].value_counts(normalize=True).sort_index()
target_df = pd.DataFrame({"p0q0": target_counts.index, "count": target_counts.values})

fig = px.bar(target_df, x="p0q0", y="count", title="Delay Frequency in Y train file",
             labels={"p0q0": "Delay (in minutes)", "count": "Frequency"})
fig.update_layout(xaxis=dict(dtick=10))
fig.show()

This graph highlights the distribution of "p0q0" in y_train_file representing the delay of trains. It helped me to remark that some "outliers" may overfit my model. Then, I decided to remove the outliers.

# Data & Engineering

### Train Data

To properly evaluate the model’s performance, the training data was split into a training set and a validation set using two approaches: train_test_split from scikit-learn and manual slicing (90%/10%).

Both ensure that the model can be assessed on unseen data before final evaluation.
The choice of using both strategies was helpful for flexibility during development, and allowed testing different experimental setups before submitting.

In [5]:
###### Train through Sklearn #####

x_train, x_val, y_train, y_val = train_test_split(x_train_file, y_train_file, test_size=0.1, random_state=54)
y_train = y_train.iloc[:, -1]
y_val = y_val.iloc[:, -1]

In [ ]:
##### Train through a simple way #####

split_idx = int(len(x_train_file) * 0.9)
x_train, x_val = x_train_file.iloc[:split_idx], x_train_file.iloc[split_idx:]
y_train, y_val = y_train_file.iloc[:split_idx], y_train_file.iloc[split_idx:]

y_train = y_train.iloc[:, -1]
y_val = y_val.iloc[:, -1]

### Outliers & Encoding

This section focuses on selecting and transforming features to improve the model’s ability to learn meaningful patterns.

Rather than using all available columns, only a subset of features (p2q0, p3q0, p4q0, p0q2, p0q3, p0q4) was retained — these represent historical delay signals and are expected to have strong predictive power.

Additionally, categorical features like gare and arret were encoded using frequency encoding.
This method was chosen over one-hot encoding or label encoding because it preserves the ordinal nature of feature importance while being computationally efficient, especially with high-cardinality features.

These transformations were essential for preparing the data to be digestible by tree-based models such as XGBoost.

In [6]:
##### Encoding #####

features_to_keep = ["p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]
X_train = x_train[features_to_keep].copy()
X_val = x_val[features_to_keep].copy()
X_test = x_test_file[features_to_keep].copy()
Y_train = y_train.copy()

# Feature Engineering: Encode "gare" and "arret" column
gare_counts = x_train['gare'].value_counts()
X_train['gare_encoded'] = x_train['gare'].map(gare_counts)
X_val['gare_encoded'] = x_val['gare'].map(gare_counts).fillna(0)
X_test['gare_encoded'] = x_test_file['gare'].map(gare_counts).fillna(0)

arret_counts = x_train['arret'].value_counts()
X_train['arret_encoded'] = x_train['arret'].map(arret_counts)
X_val['arret_encoded'] = x_val['arret'].map(arret_counts).fillna(0)
X_test['arret_encoded'] = x_test_file['arret'].map(arret_counts).fillna(0)

The following code was just to test if the date variable could bring relevant information. This try was not a success.

In [ ]:
# ## Try ##
# arret_counts = x_train['date'].value_counts()
# X_train['date_encoded'] = x_train['date'].map(arret_counts)
# X_val['date_encoded'] = x_val['date'].map(arret_counts).fillna(0)
# X_test['date_encoded'] = x_test_file['date'].map(arret_counts).fillna(0)

Before training the model, outliers were removed from the training set using a custom function based on quantile thresholds.

The motivation behind this step is to reduce the influence of extreme delay values that could distort model training and result in overfitting.
Removing such anomalies helps the model generalize better to typical delay patterns and avoid being overly sensitive to rare, unrepresentative events in the data.
This step was guided by the initial visualization of the target distribution, which revealed a long-tailed structure.

In [7]:
##### Remove Outliers #####

## First way to remove the outliers ##
# X_train_clean, Y_train_clean = remove_outliers(X_train, Y_train)

## An other way to remove the outliers ##
X_train_clean, Y_train_clean = remove_high_w_model(X_train, Y_train)

# Ensure validation set has same columns
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [8]:
# Get the frequency of each unique value of p0q0
target_counts = Y_train_clean.value_counts(normalize=True).sort_index()
target_df = pd.DataFrame({"p0q0": target_counts.index, "count": target_counts.values})

fig = px.bar(target_df, x="p0q0", y="count", title="Delay Frequency in Y train file",
             labels={"p0q0": "Delay (in minutes)", "count": "Frequency"})
fig.update_layout(xaxis=dict(dtick=10))
fig.show()

After the outliers removal, one can now remark that the distribution looks better and easily usable

In [ ]:
# #######################################################
# ##### OLD VERSION OF OUTLIERS // DO NOT RUN !!!!! #####
# #######################################################

# outlier_features = ["p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]
# x_train = remove_top_1_percent_outliers(x_train, outlier_features)
# y_train = y_train.loc[x_train.index]

# Weighted Average Model

As a simple baseline, I implemented a weighted average model that combines the available historical features.

This approach assumes that past delays — especially at previous stops or for previous trains — contain predictive information about upcoming delays. We applied specific weights to features like p2q0, p3q0 and p4q0 to simulate a decaying influence of time.

While this model is not designed for high accuracy, it serves as a reference point to measure how much more powerful machine learning models can be. It also highlights how important historical delays are, even in the absence of complex training.

In [9]:
## First method with weighted mean

x_test_final_test = x_test_file.copy()
x_test_final_test["p0q0"] = (0.6 * x_test_final_test["p0q2"] + 0.3 * x_test_final_test["p0q3"] + 0.1 * x_test_final_test["p0q4"]).round(0)
y_first_test = x_test_final_test["p0q0"]
y_first_test.to_csv("Weighted_Average.csv")

# Random Forest Model

We trained a Random Forest Regressor as one of our first model. Random Forest is particularly appealing because of its robustness to overfitting and its ability to handle non-linear relationships. The model was trained on selected features, including frequency-encoded categorical variables. We encountered some challenges related to categorical encoding, which required preprocessing to ensure all features were numerical.

Although the model performed decently, its inference time and limitations in capturing sequential patterns made it slightly less competitive compared to boosting methods.

### Fit the model

In [11]:
# Initialise the model
rf = RandomForestRegressor(n_estimators=100, random_state=54)

# Train the model
rf.fit(X_train_clean, Y_train_clean.values.ravel())

# Predecit on all values
y_pred = rf.predict(X_val)

# Evaluate the model
mae = mean_absolute_error(y_val, y_pred)
print(f"Mean Absolute Error (MAE) : {mae}")

Mean Absolute Error (MAE) : 0.7306627818616631


### Create submission file

In [ ]:
rf_full = RandomForestRegressor(n_estimators=100, random_state=54)

rf_full.fit(
    pd.concat([X_train_clean, X_val], axis=0),
    pd.concat([Y_train_clean, y_val], axis=0).values.ravel()
)

y_test_pred = rf_full.predict(X_test)
submission = pd.DataFrame({'p0q0': y_test_pred})
submission.to_csv("submission_RF.csv", index=False)

# HGB Model

Next, I tested the HistGradientBoostingRegressor (HGB) from scikit-learn, which offers a fast and scalable implementation of gradient boosting with histogram binning. This model yielded better results than Random Forest due to its ability to more finely capture complex patterns. We configured the model with a high number of iterations and a relatively deep tree structure.

Although this improved accuracy, the model still struggled to outperform XGBoost in terms of fine-tuned precision. Nevertheless, it served as a valuable middle-ground between simplicity and performance, especially given its native support for missing values and fast training time.

### Fit the model

In [ ]:
# Boosted gradient model
hgb = HistGradientBoostingRegressor(
    max_iter=1000,
    max_depth=11,
    learning_rate=0.2,
    max_bins=255,
    l2_regularization=1,
    random_state=42
)

hgb.fit(X_train_clean, Y_train_clean)

y_pred = hgb.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
print(f"Mean Absolute Error (MAE) : {mae}")

Mean Absolute Error (MAE) : 0.6978729683396774


### Create submission file

In [15]:
# Boosted gradient model
hgb_full = HistGradientBoostingRegressor(
    max_iter=1000,
    max_depth=11,
    learning_rate=0.2,
    max_bins=255,
    l2_regularization=1,
    random_state=42)

hgb_full.fit(
    pd.concat([X_train_clean, X_val], axis=0),
    pd.concat([Y_train_clean, y_val], axis=0)
)

# Predict the test file
y_test_pred = hgb_full.predict(X_test)

# Create the good file
submission = pd.DataFrame({
    "Unnamed: 0": y_sample["Unnamed: 0"],
    "p0q0": y_test_pred
})

submission.to_csv("submission_hgb.csv", index=False)

# XGBoost Model

We trained an XGBoost model using manually selected hyperparameters, chosen based on intuition and prior experience. This model already outperformed the Random Forest and HGB models thanks to its ability to perform boosting, regularization, and feature subsampling. We included frequency-encoded categorical variables and delay features.

This XGBoost version served as a strong baseline before introducing automated hyperparameter tuning. Even without Optuna, it demonstrated solid predictive power and was able to generalize well on unseen validation data.

### Fit the model

In [16]:
# Initialize the model
xgb_model = xgb.XGBRegressor(
    n_estimators=1400,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.009,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    reg_alpha=1,      # L1 (lasso)
    reg_lambda=2,     # L2 (ridge)
    random_state=54,
    n_jobs=-1           # Use all CPU cores
)

# Fit the model
xgb_model.fit(
    X_train_clean,
    Y_train_clean,
)

# Predict
y_pred = xgb_model.predict(X_val)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)

# Evaluate
mae = mean_absolute_error(y_val, y_pred)
print(f"Mean Absolute Error (MAE) : {mae}")

Mean Absolute Error (MAE) : 0.6356497369880258


### Create the submission file

In [17]:
# Predict on test

# Initialize the model
xgb_model_full = xgb.XGBRegressor(
    n_estimators=1400,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.009,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    reg_alpha=1,      # L1 (lasso)
    reg_lambda=2,     # L2 (ridge)
    random_state=54,
    n_jobs=-1           # Use all CPU cores
)

xgb_model_full.fit(
    pd.concat([X_train_clean, X_val], axis=0),
    pd.concat([Y_train_clean, y_val], axis=0),
)
y_test_pred = xgb_model.predict(X_test)
y_test_pred = y_test_pred.round(0).astype(int)
y_test_pred = pd.DataFrame(y_test_pred)

# Save in a csv file
y_test_pred.to_csv("submission_xgboost18.csv")

print("Submission file saved : submission_xgboost18.csv")

Submission file saved : submission_xgboost18.csv


# XGBoost Model with optimized parameters

### Search the best parameters and plot results

To get the best possible performance from the XGBoost model, Optuna was used for automated hyperparameter tuning via Bayesian optimization.

The objective function was defined to minimize the Mean Absolute Error (MAE) on the validation set. Rather than relying on manual tuning or grid search, Optuna dynamically explores the hyperparameter space based on past trials.
This approach is both computationally efficient and effective at discovering configurations that improve model generalization.
Using Optuna allowed us to balance depth, learning rate, regularization, and subsampling strategies in a principled way.

### Search the best parameters with Optuna

In [18]:
objective_with_data = partial(
    objective,
    X_train_clean=X_train_clean,
    Y_train_clean=Y_train_clean,
    X_val=X_val,
    y_val=y_val
)

study = optuna.create_study(direction="minimize")
study.optimize(objective_with_data, n_trials=30)

print("Best Hyperparameters :")
print(study.best_params)

[I 2025-04-04 10:31:57,092] A new study created in memory with name: no-name-dead6b12-7f8f-4af7-be4e-b2cd2f124fd2
[I 2025-04-04 10:32:07,609] Trial 0 finished with value: 0.6309290092466318 and parameters: {'max_depth': 10, 'learning_rate': 0.041935066380995055, 'subsample': 0.8873056671533177, 'colsample_bytree': 0.761196131398466, 'reg_alpha': 1.8323221138151529, 'reg_lambda': 0.7360351477605331}. Best is trial 0 with value: 0.6309290092466318.
[I 2025-04-04 10:32:17,219] Trial 1 finished with value: 0.6410148815322133 and parameters: {'max_depth': 6, 'learning_rate': 0.045417514504249946, 'subsample': 0.9311159688019296, 'colsample_bytree': 0.7763496797646326, 'reg_alpha': 2.6855215159425514, 'reg_lambda': 0.1855064528731687}. Best is trial 0 with value: 0.6309290092466318.
[I 2025-04-04 10:32:31,064] Trial 2 finished with value: 0.6406851799121794 and parameters: {'max_depth': 8, 'learning_rate': 0.015398305633881884, 'subsample': 0.6870219232666345, 'colsample_bytree': 0.963749353

Best Hyperparameters :
{'max_depth': 15, 'learning_rate': 0.02629139925597125, 'subsample': 0.7808793737037325, 'colsample_bytree': 0.8553771673579692, 'reg_alpha': 3.0864428559334, 'reg_lambda': 3.209726325514601}


In [19]:
## Plot the results of the parameters optimization

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()

### Train the model

To further improve the model’s accuracy, I implemented an exhaustive search strategy to evaluate the impact of different subsets of variables.

The main idea was to train an XGBoost model using every possible combination of minimum 4 variables among the fully encoded dataset. For each combination, I computed both the raw and rounded Mean Absolute Error (MAE) on the validation set to reflect how the model would behave in real prediction settings. The motivation behind this experiment was to identify whether a smaller, more focused set of variables could outperform the full feature set by reducing noise or redundancy. I expected to discover some lean combinations that retained only the most predictive signals.

However, the results showed that no combination significantly outperformed the model I used at the beginning so I decided to keep the initial set of variables.

In [ ]:
# # Liste de toutes les colonnes encodées disponibles
# all_features = X_train_clean.columns.tolist()

# results = []

# # Boucle sur toutes les combinaisons possibles (au moins 2 variables)
# for r in range(4, len(all_features)+1):
#     for subset in combinations(all_features, r):
#         selected_features = list(subset)

#         # Préparer les DMatrix
#         dtrain = xgb.DMatrix(X_train_clean[selected_features], label=Y_train_clean)
#         dval = xgb.DMatrix(X_val[selected_features], label=y_val)

#         # Entraîner le modèle
#         model = xgb.train(
#             study.best_params,
#             dtrain,
#             num_boost_round=2000,
#             evals=[(dval, "validation")],
#             early_stopping_rounds=50,
#             verbose_eval=False
#         )

#         # Prédictions et MAE
#         preds = model.predict(dval).round().astype(int)
#         mae = mean_absolute_error(y_val, preds)

#         results.append({
#             "features": selected_features,
#             "mae": mae
#         })

# # Convertir en DataFrame et afficher les 3 meilleurs
# results_df = pd.DataFrame(results).sort_values(by="mae")
# top_3 = results_df.head(3)

# print("🔝 Top 3 feature combinations with lowest MAE:")
# for i, row in top_3.iterrows():
#     print(f"MAE: {row['mae']:.4f} | Features: {row['features']}")

## Results ##

# 🔝 Top 3 feature combinations with lowest MAE:
# MAE: 0.6261 | Features: ['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'gare_encoded', 'arret_encoded', 'date_encoded']
# MAE: 0.6269 | Features: ['p2q0', 'p3q0', 'p4q0', 'gare_encoded', 'arret_encoded', 'date_encoded']
# MAE: 0.6276 | Features: ['p2q0', 'p3q0', 'p4q0', 'p0q2', 'gare_encoded', 'arret_encoded', 'date_encoded']

🔝 Top 3 feature combinations with lowest MAE:
MAE: 0.6261 | Features: ['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'gare_encoded', 'arret_encoded', 'date_encoded']
MAE: 0.6269 | Features: ['p2q0', 'p3q0', 'p4q0', 'gare_encoded', 'arret_encoded', 'date_encoded']
MAE: 0.6276 | Features: ['p2q0', 'p3q0', 'p4q0', 'p0q2', 'gare_encoded', 'arret_encoded', 'date_encoded']


Now that we have the best parameters we can fit the model and get an internal MAE with the validation set.

In [20]:
best_params = study.best_params
best_params.update({
    "objective": "reg:squarederror",
    "eval_metric": "mae",
    "n_jobs": -1,
    "random_state": 42
})

dtrain = xgb.DMatrix(X_train_clean, label=Y_train_clean)
dval = xgb.DMatrix(X_val, label=y_val)

best_xgboost_model = xgb.train(
    best_params,
    dtrain,
    num_boost_round=2000,
    evals=[(dval, "validation")],
    early_stopping_rounds=50,
    verbose_eval=True,
)

y_pred = best_xgboost_model.predict(dval).round().astype(int)

[0]	validation-mae:0.89388
[1]	validation-mae:0.88480
[2]	validation-mae:0.87775
[3]	validation-mae:0.87172
[4]	validation-mae:0.86541
[5]	validation-mae:0.86147
[6]	validation-mae:0.85744
[7]	validation-mae:0.85358
[8]	validation-mae:0.84823
[9]	validation-mae:0.84368
[10]	validation-mae:0.83935
[11]	validation-mae:0.83463
[12]	validation-mae:0.83053
[13]	validation-mae:0.82750
[14]	validation-mae:0.82347
[15]	validation-mae:0.81932
[16]	validation-mae:0.81650
[17]	validation-mae:0.81204
[18]	validation-mae:0.80950
[19]	validation-mae:0.80564
[20]	validation-mae:0.80259
[21]	validation-mae:0.80032
[22]	validation-mae:0.79686
[23]	validation-mae:0.79371
[24]	validation-mae:0.79108
[25]	validation-mae:0.78786
[26]	validation-mae:0.78504
[27]	validation-mae:0.78344
[28]	validation-mae:0.78053
[29]	validation-mae:0.77786
[30]	validation-mae:0.77503
[31]	validation-mae:0.77222
[32]	validation-mae:0.76922
[33]	validation-mae:0.76689
[34]	validation-mae:0.76448
[35]	validation-mae:0.76248
[3

### Error plots

In [21]:
## Print the distribution of the error ##

errors = y_val - y_pred
fig = px.histogram(errors, nbins=100, title="Error Distribution (y_val - y_pred)")
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(xaxis_title="Error", yaxis_title="Number of occurences")
fig.show()

## Print the distribution of the error in absolute value ##

abs_errors = np.abs(y_val - y_pred)
fig = px.histogram(abs_errors, nbins=100, title="Absolute Error Distribution (y_val - y_pred)")
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(xaxis_title="Absolute Error", yaxis_title="Number of occurences")
fig.show()

The idea behind the generation of the graphs above was to detect if all the errors were negative or all positive in order to maybe adjust the model. It highlights that some errors are way above the others and only in the negative part.

### Variables importance

In [22]:
importance = best_xgboost_model.get_score(importance_type='gain')
importance_df = pd.DataFrame({
    "feature": list(importance.keys()),
    "importance": list(importance.values())
}).sort_values(by="importance", ascending=False).head(10)

fig = px.bar(importance_df, x="importance", y="feature", orientation="h",
             title="Top 10 most important variables (gain)")
fig.update_layout(yaxis_title="Variables", xaxis_title="Gain")
fig.show()

This graph shows the importance of each variable. It helped me to understand which variables are the most important for my model.

### Create the submission file

Once the best hyperparameters were identified using Optuna, the model was retrained on the entire available dataset (training + validation) to maximize the data used for learning.

This ensures the model has access to all historical examples and captures as much pattern richness as possible.

Importantly, early_stopping_rounds was removed, since no separate validation set was available at this stage — and the optimal number of boosting rounds discovered during Optuna tuning was reused with a small buffer. This step is critical to building the final model that will be used for test set predictions.

The trained final model was then used to generate predictions on the test set.

Care was taken to ensure the test set had the same feature structure as the training data (via column reindexing). Predictions were rounded to the nearest integer to match the target variable format (p0q0 being in whole minutes). The results were then exported to a CSV file, formatted according to the submission requirements.

In [23]:
X_full = pd.concat([X_train_clean, X_val], axis=0)
Y_full = pd.concat([Y_train_clean, y_val], axis=0)
dtrain_full = xgb.DMatrix(X_full, label=Y_full)
X_test = X_test.reindex(columns=X_train_clean.columns, fill_value=0)
dtest = xgb.DMatrix(X_test)

xgb_opti_full_model = xgb.train(
    best_params,
    dtrain_full,
    num_boost_round=best_xgboost_model.best_iteration + 50 if "best_xgboost_model" in locals() else 1000
)

# Predictions
y_test_pred = xgb_opti_full_model.predict(dtest)
y_test_pred = y_test_pred.round(0).astype(int)

y_test_pred = pd.DataFrame(y_test_pred, columns=["p0q0"])
y_test_pred.index.name = "index"

y_test_pred.to_csv("submission_xgboost19.csv")

print("Submission file saved : submission_xgboost19.csv")

Submission file saved : submission_xgboost19.csv


# Conclusion

In this notebook, I built a complete pipeline for predicting train delays using historical scheduling data.

We explored several machine learning models, including tree-based methods such as Random Forest, HGB, and XGBoost, with and without hyperparameter tuning. Through preprocessing, feature engineering, and careful evaluation, I iteratively improved model performance. Optuna-based optimization brought an additional performance gain by automatically fine-tuning the model. Finally, I selected the best model and retrained it on the full dataset to produce final predictions.

The project illustrates the importance of combining domain knowledge, data cleaning, model tuning, and interpretability in building a reliable predictive system for real-world transportation data.